# Forest Monitoring & Deforestation Detection
**NB 06** | Amazon Basin, Brazil | ChangeFormer + DINOv3 CHMv2

In [1]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv

client = pgv.PyGeoVision()
print(client)

from pygeovision.models.change_detection.changeformer import ChangeDetection
from pygeovision.models.foundation.dinov3 import CHMv2Model


14:51:04 INFO [      engine] PyGeoFetch ready


2026-07-15 14:51:04,070 [INFO] pygeofetch.core.engine: PyGeoFetch ready


14:51:04 INFO [      engine] PyGeoFetch ready


2026-07-15 14:51:04,073 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-15 14:51:04,075 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-15 14:51:04,076 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing


PyGeoVision(v2.1.6 | pygeofetch=✓ | pgf_v2=✓v2 | geoai=independent | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)


In [2]:
BBOX        = (-62.50, -3.80, -62.20, -3.50)  # Para state, Amazon basin
DATE_BEFORE = ('2020-07-01', '2020-09-30')
DATE_AFTER  = ('2024-07-01', '2024-09-30')
PROVIDERS   = ['planetary_computer']
BANDS       = ['B02','B03','B04','B08','B11','B12']
DATA_DIR    = pathlib.Path('./results/nb06')
DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
pre_r  = client.search(bbox=BBOX, date_range=DATE_BEFORE, providers=PROVIDERS, cloud_cover_max=20)
post_r = client.search(bbox=BBOX, date_range=DATE_AFTER,  providers=PROVIDERS, cloud_cover_max=20)
print(f"Pre: {len(pre_r)} | Post: {len(post_r)}")
PRE_DIR  = DATA_DIR / 'pre';  PRE_DIR.mkdir(exist_ok=True)
POST_DIR = DATA_DIR / 'post'; POST_DIR.mkdir(exist_ok=True)
pre_dl  = client.download(pre_r[:1],  output_dir=str(PRE_DIR),  bands=BANDS, post_process=['reproject:EPSG:32720','cog'])
post_dl = client.download(post_r[:1], output_dir=str(POST_DIR), bands=BANDS, post_process=['reproject:EPSG:32720','cog'])
pre_scene  = pre_dl[0].path  if pre_dl  and pre_dl[0].success  else None
post_scene = post_dl[0].path if post_dl and post_dl[0].success else None


2026-07-15 14:51:13,731 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-62.5, -3.8, -62.2, -3.5) | 2020-07-01→2020-09-30 | cloud≤20%



┌──────────────────────────────────────────────────────────────────────┐
│                          🛰  PYGEOFETCH SEARCH                        │
├──────────────────────────────────────────────────────────────────────┤
│  Providers              planetary_computer                                    │
│  BBox                   [-62.500, -3.800, -62.200, -3.500]                    │
│  Date range             2020-07-01  →  2020-09-30                             │
│  Cloud max              20.0%                                                 │
│  Product                any                                                   │
└──────────────────────────────────────────────────────────────────────┘

14:51:15 INFO [planetary_computer] Planetary Computer: 23 results


2026-07-15 14:51:15,559 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 23 results
2026-07-15 14:51:15,572 [INFO] pygeovision.data.fetch: Search complete: 23 results
2026-07-15 14:51:15,576 [INFO] pygeovision.data.fetch: Searching 1 provider(s) [planetary_computer] | bbox=(-62.5, -3.8, -62.2, -3.5) | 2024-07-01→2024-09-30 | cloud≤20%


  ✓  planetary_computer              23 scenes   1.8s

  ┌─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
  │  SCENE ID                                    DATE        SATELLITE       CLOUD   PRODUCT  POLARISATION  PASS         ORBIT  SCORE  PROVIDER              │
  ├─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
  │  LE07_L2SP_232063_20200907_02_T1             2020-09-07  landsat-7       0%      —        —            —            —      0.70   planetary_computer    │
  │  LE07_L2SP_232062_20200907_02_T1             2020-09-07  landsat-7       0%      —        —            —            —      0.70   planetary_computer    │
  │  S2A_MSIL2A_20200729T142741_R053_T20MNA_202  2020-07-29  Sentinel-2A     0%      —        —            descending   53     0.70   planetary_computer  

2026-07-15 14:51:16,874 [INFO] pygeofetch.provider.planetary_computer: Planetary Computer: 35 results
2026-07-15 14:51:16,890 [INFO] pygeovision.data.fetch: Search complete: 35 results


...

  ⠧  Downloading 1 scene(s)…  14:51:27 INFO [planetary_computer]   ✓ T20MNA_20200922T142739_B01_60m.tif (5.9 MB)


2026-07-15 14:51:27,440 [INFO] pygeofetch.provider.planetary_computer:   ✓ T20MNA_20200922T142739_B01_60m.tif (5.9 MB)


14:51:27 INFO [planetary_computer]   Fetching asset 'B02' → T20MNA_20200922T142739_B02_10m.tif


2026-07-15 14:51:27,442 [INFO] pygeofetch.provider.planetary_computer:   Fetching asset 'B02' → T20MNA_20200922T142739_B02_10m.tif


  ⠋  Downloading 1 scene(s)…  

In [ ]:
PRE_READY  = DATA_DIR / 'pre_preprocessed.tif'
POST_READY = DATA_DIR / 'post_preprocessed.tif'
for scene, out, label in [(pre_scene, PRE_READY, 'PRE'), (post_scene, POST_READY, 'POST')]:
    if scene:
        scl = next(iter(scene.parent.rglob('*SCL*.tif')), None)
        rep = client.validator.validate(str(scene), required_bands=len(BANDS))
        r   = client.prepare_for_ai(str(scene), stack_bands=BANDS, bbox=BBOX,
            scl_path=str(scl) if scl else None, normalise='scale_factor',
            scale_factor=10000.0, model_type='change_detection', output_path=str(out))
        print(f"{label}: {r['steps']}  shape={r['shape']}")


In [ ]:
# Forest loss detection
DEFOR_MAP = DATA_DIR / 'deforestation_map.tif'
if PRE_READY.exists() and POST_READY.exists():
    cd  = ChangeDetection(model_variant='changeformer', in_channels=len(BANDS))
    cd.build()
    res = cd.detect(str(PRE_READY), str(POST_READY), output_path=str(DEFOR_MAP))
    print(f"Forest change: {res['change_pct']:.2f}% of study area")


In [ ]:
# Canopy height model with DINOv3 CHMv2 on POST scene
if POST_READY.exists():
    chm   = CHMv2Model()
    chm_r = chm.predict_canopy_height(str(POST_READY))
    print(f"Canopy height — mean: {chm_r.get('mean_m',0):.1f}m  max: {chm_r.get('max_m',0):.1f}m")
    bm    = chm.estimate_biomass(str(POST_READY))
    print(f"Biomass estimate: {bm.get('mean_t_dm_ha',0):.1f} t DM/ha")


In [ ]:
# Compute NBR on both preprocessed scenes for burn/forest-loss index
if PRE_READY.exists() and POST_READY.exists():
    idx_dir = DATA_DIR / 'indices'
    idx_dir.mkdir(exist_ok=True)
    nbr_pre  = client.indices.nbr(str(PRE_READY),  nir_band=4, swir2_band=6, output_path=str(idx_dir/'nbr_pre.tif'))
    nbr_post = client.indices.nbr(str(POST_READY), nir_band=4, swir2_band=6, output_path=str(idx_dir/'nbr_post.tif'))
    import numpy as np
    import rasterio
    with rasterio.open(str(nbr_pre)) as s1, rasterio.open(str(nbr_post)) as s2:
        dnbr = s1.read(1) - s2.read(1)
        prof = s1.profile.copy()
    dnbr_path = idx_dir / 'dnbr.tif'
    prof.update(dtype='float32', compress='lzw')
    with rasterio.open(str(dnbr_path), 'w', **prof) as dst:
        dst.write(dnbr[np.newaxis])
    print(f"dNBR stats — mean={dnbr.mean():.3f}  max={dnbr.max():.3f}")
    print(f"High severity (dNBR>0.66): {(dnbr>0.66).mean()*100:.1f}%")


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=20, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=200.0, simplify_tolerance=1.0)
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB06 — Forest Monitoring** complete.
- Study area: Amazon Basin Brazil
- Sensor: Sentinel-2 bi-temporal
- Model: ChangeFormer + DINOv3 CHMv2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG